In [1]:
import pandas as pd
import numpy as np

threshold = -50
window = 48
gap_threshold = 6

In [8]:
df = pd.read_csv('allDST_omni.csv')
triggers = df[df['DST'] < threshold].index

events_indices = set()
processed_indices = set()

for idx in triggers:
    if idx in processed_indices:
        continue

    start_idx = max(0, idx - window)
    end_idx = min(len(df) - 1, idx + window)

    new_indices = range(start_idx, end_idx + 1)
    events_indices.update(new_indices)
    processed_indices.update(new_indices)

events_df = df.loc[sorted(list(events_indices))].copy()

display(events_df.head())
print(f'Pôvodný počet riadkov: {len(df)}')
print(f'Počet riadkov po extrakcii udalostí: {len(events_df)}')
print(f'Počet unikátnych udalostí (skupín): {len(triggers)}')

,Unnamed: 0,time1,bz_gsm,v,DST,DST+1,DST+2,DST+3,DST+4,DST+5,DST+6
669,669,1963-01-28 21:30:00+00:00,-0.2,285.0,7,7,8.0,6.0,8.0,7.0,9.0
670,670,1963-01-28 22:30:00+00:00,-0.2,285.0,8,8,6.0,8.0,7.0,9.0,11.0
671,671,1963-01-28 23:30:00+00:00,-0.2,285.0,6,6,8.0,7.0,9.0,11.0,12.0
672,672,1963-01-29 00:30:00+00:00,-0.2,285.0,8,8,7.0,9.0,11.0,12.0,13.0
673,673,1963-01-29 01:30:00+00:00,-0.2,285.0,7,7,9.0,11.0,12.0,13.0,12.0


Pôvodný počet riadkov: 552568
Počet riadkov po extrakcii udalostí: 143721
Počet unikátnych udalostí (skupín): 29116


In [9]:
triggers = df[df['DST'] < threshold].index
raw_windows = []

for t in triggers:
    raw_windows.append([max(0, t - window), min(len(df) - 1, t + window)])

if not raw_windows:
    events_summary_df = pd.DataFrame()
    print("Nenašli sa žiadne udalosti.")
else:
    raw_windows.sort()
    merged_windows = []
    curr_start, curr_end = raw_windows[0]

    for i in range(1, len(raw_windows)):
        next_start, next_end = raw_windows[i]

        if next_start <= (curr_end + gap_threshold):
            curr_end = max(curr_end, next_end)
        else:
            merged_windows.append([curr_start, curr_end])
            curr_start, curr_end = next_start, next_end

    merged_windows.append([curr_start, curr_end])

    summary_data = []
    for i, (start, end) in enumerate(merged_windows, 1):
        event_slice = df.loc[start:end]
        min_dst = event_slice['DST'].min()

        summary_data.append({
            'ID': i,
            'Udalost_Od': df.loc[start, 'time1'],
            'Udalost_Do': df.loc[end, 'time1'],
            'Min_DST_v_udalosti': min_dst
        })

    events_summary_df = pd.DataFrame(summary_data)
    events_summary_df.to_csv('zoznam_udalosti_final.csv', index=False)

    display(events_summary_df.head())
    print(f'Celkový počet zlúčených udalostí (s medzerou min 6h): {len(events_summary_df)}')
    print('Súbor "zoznam_udalosti_final.csv" bol aktualizovaný.')

,ID,Udalost_Od,Udalost_Do,Min_DST_v_udalosti
0,1,1963-01-28 21:30:00+00:00,1963-02-02 21:30:00+00:00,-75
1,2,1963-02-08 01:30:00+00:00,1963-02-12 09:30:00+00:00,-62
2,3,1963-06-05 01:30:00+00:00,1963-06-09 08:30:00+00:00,-78
3,4,1963-08-17 22:30:00+00:00,1963-08-22 11:30:00+00:00,-84
4,5,1963-09-12 07:30:00+00:00,1963-09-30 21:30:00+00:00,-236


Celkový počet zlúčených udalostí (s medzerou min 6h): 953
Súbor "zoznam_udalosti_final.csv" bol aktualizovaný.
